In [ ]:
# Getting imports and the model setup
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")
import yaml
import importlib.resources
from smolagents import OpenAIModel, InferenceClientModel, LogLevel

from dotenv import load_dotenv
load_dotenv()

# model_name = "gpt-4o"
model_name = "gpt-5.4-mini"
# model_name = "Qwen/Qwen3.7-Plus"
# model_name = "Qwen/Qwen3.5-9B"

enable_thinking = False  # hardcoded (was interactive input()) so this notebook can run headlessly via nbconvert

if model_name in ["gpt-4o", "gpt-5.4-mini"]:
    model = OpenAIModel(
        model_id=model_name,
        api_key=os.environ["OPENAI_API_KEY"],
    )
elif model_name in ["Qwen/Qwen3.7-Plus", "Qwen/Qwen3.5-9B", "deepseek-ai/DeepSeek-R1", "openai/gpt-oss-120b"]:
    # Together requires enable_thinking nested inside chat_template_kwargs for vLLM-served open-weight
    # models -- a bare top-level enable_thinking happened to also work for Qwen3.7-Plus but was silently
    # ignored for Qwen3.5-9B. Verified chat_template_kwargs works correctly (both True and False) for both.
    # client_kwargs timeout bounds a single API call so a stalled/hanging stream fails within 5 minutes
    # instead of hanging indefinitely -- evaluate_agent already catches and records such errors.
    model = OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/", # Leave this blank to query OpenAI servers.
        api_key=os.environ["TOGETHER_API_KEY"], # Switch to the API key for the server you're targeting.
        extra_body={"chat_template_kwargs": {"enable_thinking": enable_thinking}},
        client_kwargs={"timeout": 300.0},
    )
print(f"Using model: {model_name} with thinking enabled: {enable_thinking}")

In [6]:
# Debug check: fail loudly the moment any step returns reasoning despite enable_thinking=False,
# rather than only noticing it later by eyeballing console output.
from common_setup import assert_no_reasoning

In [7]:
# Getting the tools setup and the agent setup
from smolagents import CodeAgent
from smolagents.monitoring import LogLevel
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

agent = CodeAgent(
    tools=tools,
    model=model,
    max_steps=50,
    verbosity_level=LogLevel.DEBUG,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    stream_outputs=("deepseek" in model_name) or ("Qwen" in model_name),
    step_callbacks=[assert_no_reasoning] if not enable_thinking else None,
)

In [8]:
# Load GAIA validation set from HuggingFace
# Visit https://huggingface.co/datasets/gaia-benchmark/GAIA to request access first
import pandas as pd
from common_setup import load_gaia_dataset

SET_TO_RUN = "validation"
eval_ds = load_gaia_dataset(set_to_run=SET_TO_RUN)

print(f"Loaded {len(eval_ds)} examples")
print(pd.DataFrame(eval_ds)["task"].value_counts())

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 165 examples
task
2    86
1    53
3    26
Name: count, dtype: int64


**Note on token counts below:** `step.token_usage` is per-step (that single LLM call's own `input_tokens`/`output_tokens`, taken straight from the API's `usage` field), not a running total. Because the full step history is resent every call, `input_tokens` still grows step-over-step on its own — it's just not incremental. This differs from the smolagents console trajectory (and `agent.monitor.get_total_token_counts()`), which print a *cumulative sum* of each step's `token_usage` and will not match these per-step prints after step 1. See the "Token usage accounting" section in CLAUDE.md for details.

In [9]:
# Scoring + eval loop now live in common_setup.py, shared with markovReAct.ipynb
from common_setup import evaluate_agent, question_scorer

In [10]:
print(model.kwargs)
print(agent.model.kwargs)
print(agent.model is model)

{}
{}
True


In [ ]:
results = evaluate_agent(agent, eval_ds, ti_tool, visualizer, n_samples=165, output_file=f"naive_react_{model_name}_{enable_thinking}_fixedbrowser.jsonl", pickle_dir=f"naive_react_{model_name}_{enable_thinking}_fixedbrowser")

In [8]:
df = pd.DataFrame(results)
total = len(df)
correct = df["is_correct"].sum()

print(f"=== GAIA Evaluation Results ===")
print(f"Overall accuracy:   {correct}/{total} = {correct/total:.1%}")
print(f"Avg time per question: {df['time_taken_seconds'].mean():.1f}s")
print(f"Avg steps per question: {df['num_steps'].mean():.1f}")

total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0)).mean()
print(f"Total tokens used:  {total_tokens:,}")

print(f"\nAccuracy by level:")
for level in sorted(df["task"].unique()):
    level_df = df[df["task"] == level]
    lc = level_df["is_correct"].sum()
    lt = len(level_df)
    print(f"  Level {level}: {lc}/{lt} = {lc/lt:.1%}")

print(f"\nTool usage (total calls across all questions):")
tool_usage_df = pd.DataFrame(df["tool_usage"].tolist()).sum().sort_values(ascending=False)
for tool, count in tool_usage_df.items():
    if count > 0:
        print(f"  {tool}: {int(count)}")

=== GAIA Evaluation Results ===
Overall accuracy:   83/165 = 50.3%
Avg time per question: 112.5s
Avg steps per question: 18.9
Total tokens used:  467,797.3393939394

Accuracy by level:
  Level 1: 34/53 = 64.2%
  Level 2: 43/86 = 50.0%
  Level 3: 6/26 = 23.1%

Tool usage (total calls across all questions):
  web_search: 1086
  visit_page: 671
  page_down: 627
  find_on_page_ctrl_f: 354
  final_answer: 154
  page_up: 78
  inspect_file_as_text: 59
  find_archived_url: 30
  visualizer: 19
  find_next: 14
